## Combined-All

In [1]:
# railway_prb_simulation_per_cell_v5_naive_vs_rl.py
#
# - Reactive baseline = your v4 naive_reactive (same minislot scheduling + puncturing)
# - Proactive = RL chooses quota shares (DL/UL), everything else identical to v4
# - RL state = current demand + forecast(t+1) from STGMFiLMFormer
# - RL training = 50 episodes; evaluation metrics collected AFTER training only
# - Comparison plots: delay(ms), throughput(DL/UL), violation rate(%)
#
# Dependencies:
#   pip install pandas numpy torch matplotlib gymnasium stable-baselines3
#
# Run:
#   python railway_prb_simulation_per_cell_v5_naive_vs_rl.py

from __future__ import annotations

import ast
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
import os
import psutil
from mfilm_sim_forecaster import STGMFiLMForecaster

try:
    from IPython.display import display
except Exception:
    display = None

try:
    import gymnasium as gym
    from gymnasium import spaces
    from stable_baselines3 import PPO
    from stable_baselines3.common.monitor import Monitor
    from stable_baselines3.common.callbacks import BaseCallback
except Exception:
    gym = None
    spaces = None
    PPO = None
    Monitor = None
    BaseCallback = object

# ========================= SCENARIO CONFIG =========================
# Select the simulation dataset scenario here.
# Available inside STG_Datasets/: STGdata_A, STGdata_B, STGdata_C
DATASET_BUNDLE_ROOT = Path("STG_Datasets")
SCENARIO_NAME = "STGdata_C"

SCENARIO_ROOT = DATASET_BUNDLE_ROOT / SCENARIO_NAME
FINAL_DATASET_CSV = SCENARIO_ROOT / "data" / f"{SCENARIO_NAME}.csv"
CELL_GNB_PATH     = DATASET_BUNDLE_ROOT / "cell_gnb_relation.csv"
TRAINS_FOLDER     = SCENARIO_ROOT / "input_trains"

# Forecast checkpoint path is independent from the selected scenario.
# Keep the filename as-is even when switching between STGdata_A/B/C.
FORECAST_CKPT     = Path("forecasting_results_missingData/checkpoints/STGdata_C_STG_MFilmFormer_best.pt")
USE_FORECAST      = True

# Enable only the approaches you want to run during a trial.
RUN_APPROACHES = {
    "Reactive": True,
    "Proactive": False,
    "ProReactive_heuristic": False,
    "OurPRISR_ProReactive_DRL": True,
}

PROREACTIVE_PRED_WEIGHT = 0.5  # weight for t+1 prediction in quota calculation
PROREACTIVE_W_ERR = 0.3
PROREACTIVE_W_BACKLOG = 0.7
PROREACTIVE_CONF_THRESHOLD = 0.6
PROREACTIVE_BACKLOG_SCALE = 5.0
PROREACTIVE_EMA_ALPHA = 0.8

# DRL (OurPRISR_ProReactive_DRL)
DRL_TRAIN_EPISODES = 100
IF_TRAINING = False
DRL_MODEL_PATH = Path("saved_models/proreactive_drl_ppo.zip")
DRL_TRAINING_REWARD_PLOT = "drl_training_rewards.png"
DRL_DELAY_TARGET_CRIT_MS = 10.0
DRL_DELAY_TARGET_PERF_MS = 30.0
DRL_MAX_TRAIN_STEPS_PER_EPISODE = 24

# Stop conditions (use one or both)
END_TIME_S: Optional[float] = 400.0   # simulate up to this time (seconds). Set None to ignore.
END_STEP: Optional[int] = None       # simulate up to this step. Set None to ignore.
START_TIME_S: float = 0.0

# ========================= RADIO CONFIG =========================
K_PRB_DL = 25
K_PRB_UL = 25

EPISODE_S = 0.250
MINISLOT_S = 0.0005
MINISLOTS_PER_EPISODE = int(round(EPISODE_S / MINISLOT_S))  # 500
MINISLOT_THR_SCALE = 1

# ========================= METRICS HELPERS =========================
_psutil_proc = psutil.Process(os.getpid())

# Find first available RAPL energy counter (Linux)
_RAPL_ENERGY_PATH = None
_ENERGY_AVAILABLE = None
try:
    base = Path("/sys/class/powercap")
    if base.exists():
        for p in sorted(base.glob("intel-rapl:*/energy_uj")):
            _RAPL_ENERGY_PATH = p
            break
        if _RAPL_ENERGY_PATH is None:
            for p in sorted(base.glob("**/energy_uj")):
                _RAPL_ENERGY_PATH = p
                break
except Exception:
    _RAPL_ENERGY_PATH = None
_ENERGY_AVAILABLE = None


def _read_rapl_uj() -> float:
    if _RAPL_ENERGY_PATH is None:
        return float("nan")
    try:
        return float(Path(_RAPL_ENERGY_PATH).read_text().strip())
    except Exception:
        return float("nan")


def _read_rss_mb() -> float:
    try:
        return _psutil_proc.memory_info().rss / (1024.0 * 1024.0)
    except Exception:
        return float("nan")

# kb -> bits conversion (v4 style)
KB_TO_BITS = 1000.0 * 8.0

# ========================= COLUMN NAMES =========================
TIME_COL = "time_s"
STEP_COL = "step"
CI_COL = "CI"
TRAIN_IDS_COL = "train_ids"

CQI_SB_COLS = [f"CQI_sb{i}" for i in range(1, 8)]  # CQI_sb1..CQI_sb7

# ========================= APP DEMANDS =========================
APP_DEFS = {
    "voice": dict(dl="voice_DL_kb", ul="voice_UL_kb", slice="critical"),
    "etcs": dict(dl="etcs_DL_kb", ul="etcs_UL_kb", slice="critical"),
    "ato": dict(dl="ato_DL_kb", ul="ato_UL_kb", slice="critical"),
    "remote_engine_ctrl": dict(dl="remote_engine_ctrl_DL_kb", ul="remote_engine_ctrl_UL_kb", slice="critical"),
    "pub_warn": dict(dl="pub_warn_DL_kb", ul="pub_warn_UL_kb", slice="critical"),
    "telemetry_nc": dict(dl="telemetry_nc_DL_kb", ul="telemetry_nc_UL_kb", slice="performance"),
    "equip_ctl": dict(dl="equip_ctl_DL_kb", ul="equip_ctl_UL_kb", slice="performance"),
    "asset_tel": dict(dl="asset_tel_DL_kb", ul="asset_tel_UL_kb", slice="performance"),
    "pis": dict(dl="pis_DL_kb", ul="pis_UL_kb", slice="performance"),
    "video_surv": dict(dl="video_surv_DL_kb", ul="video_surv_UL_kb", slice="performance"),
}

# Per-app packetization profile used to split episode volume into intra-episode jobs.
# Units: payload_bits in bits, interval_ms/pdb_ms in milliseconds.
APP_PACKET_PROFILE = {
    "voice":              {"payload_bits": 800.0,    "interval_ms": 20.0, "pdb_ms": 65.0},
    "etcs":               {"payload_bits": 1600.0,   "interval_ms": 20.0, "pdb_ms": 50.0},
    "ato":                {"payload_bits": 1600.0,   "interval_ms": 20.0, "pdb_ms": 50.0},
    "remote_engine_ctrl": {"payload_bits": 120000.0, "interval_ms": 25.0, "pdb_ms": 50.0},
    "pub_warn":           {"payload_bits": 1600.0,   "interval_ms": 20.0, "pdb_ms": 65.0},
    "telemetry_nc":       {"payload_bits": 1600.0,   "interval_ms": 20.0, "pdb_ms": 190.0},
    "equip_ctl":          {"payload_bits": 1600.0,   "interval_ms": 20.0, "pdb_ms": 50.0},
    "asset_tel":          {"payload_bits": 1600.0,   "interval_ms": 20.0, "pdb_ms": 190.0},
    "pis":                {"payload_bits": 1600.0,   "interval_ms": 20.0, "pdb_ms": 280.0},
    "video_surv":         {"payload_bits": 120000.0, "interval_ms": 25.0, "pdb_ms": 280.0},
}

SLICES = ["critical", "performance"]
LINKS = ["DL", "UL"]

# ========================= PHY (same as your v4) =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

J = 1
v_Llayers = 1
OH = 0.14
f = 1
mu = 0
T_s = 1e-3 / (7 * 2**mu)

CQI_TO_MCS = {
    1: 2, 2: 2, 3: 2, 4: 4, 5: 4,
    6: 4, 7: 4, 8: 6, 9: 6, 10: 6,
    11: 6, 12: 6, 13: 8, 14: 8, 15: 8
}
CQI_TO_Rmax = {
    1: 0.07617188,
    2: 0.18847656,
    3: 0.43847656,
    4: 0.36914063,
    5: 0.47851563,
    6: 0.6015625,
    7: 0.45507813,
    8: 0.55371094,
    9: 0.65039063,
    10: 0.75390625,
    11: 0.85253906,
    12: 0.69433594,
    13: 0.77832031,
    14: 0.86425781,
    15: 0.92578125,
}

# ========================= DATA STRUCTURES =========================
@dataclass
class Job:
    job_id: Tuple[int, str, str, int]  # (train_id, app_name, link, seq)
    train_id: int
    app_name: str
    link: str           # "DL" or "UL"
    slice_name: str     # "critical" or "performance"
    arrival_time_s: float
    remaining_bits: float
    total_bits: float
    first_served_time_s: Optional[float] = None
    done_time_s: Optional[float] = None
    dropped_time_s: Optional[float] = None

# ========================= HELPERS =========================
def _init_slice_link_dict(default=0.0) -> Dict[Tuple[str, str], float]:
    return {(s, l): float(default) for s in SLICES for l in LINKS}

def prb_to_subband_idx(prb: int) -> int:
    return prb // 4 if prb < 24 else 6

def compute_old_thr_from_cqi(cqi: int) -> float:
    cqi = int(np.clip(int(cqi), 1, 15))
    R_max = CQI_TO_Rmax.get(cqi, 0.0)
    Q_m = CQI_TO_MCS.get(cqi, 0.0)
    thr = (1e-3 * J * v_Llayers * Q_m * f * R_max * (12 / T_s) * (1 - OH))
    return float(thr)

def build_gamma_bits_per_prb(row: pd.Series, k_prb: int) -> np.ndarray:
    cqi_sbs = []
    for col in CQI_SB_COLS:
        v = row.get(col, np.nan)
        if pd.isna(v):
            cqi_sbs.append(1)
        else:
            cqi_sbs.append(int(np.clip(int(round(float(v))), 1, 15)))

    gamma = np.zeros(k_prb, dtype=float)
    for prb in range(k_prb):
        sb = prb_to_subband_idx(prb)
        cqi = cqi_sbs[sb]
        gamma[prb] = MINISLOT_THR_SCALE * compute_old_thr_from_cqi(cqi)
    return gamma

def parse_train_ids(x) -> List[int]:
    if isinstance(x, list):
        return [int(v) for v in x]
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "" or s == "[]":
        return []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return [int(t) for t in v]
    except Exception:
        pass
    nums = re.findall(r"\d+", s)
    return [int(n) for n in nums]

def pick_job_largest_backlog(jobs: List[Job]) -> Optional[Job]:
    best = None
    best_rem = -1.0
    for jb in jobs:
        if jb.remaining_bits > best_rem:
            best_rem = jb.remaining_bits
            best = jb
    return best


def pick_job_fcfs(jobs: List[Job]) -> Optional[Job]:
    if not jobs:
        return None
    return min(jobs, key=lambda j: (j.arrival_time_s, j.job_id[3]))

def compute_quota_prbs(demand_crit_bits: float, demand_perf_bits: float, total_prbs: int) -> Tuple[int, int]:
    d_tot = demand_crit_bits + demand_perf_bits
    if d_tot <= 0:
        return 0, total_prbs
    q_crit = int(round(total_prbs * (demand_crit_bits / d_tot)))
    q_crit = max(0, min(total_prbs, q_crit))
    q_perf = total_prbs - q_crit
    return q_crit, q_perf

def shares_to_quota(share_crit: float, total_prbs: int, d_crit: float, d_perf: float) -> Tuple[int,int]:
    """
    RL chooses share. Add guardrails so we don't waste PRBs when slice has zero demand.
    """
    if d_crit <= 0 and d_perf <= 0:
        return 0, total_prbs
    if d_crit <= 0:
        return 0, total_prbs
    if d_perf <= 0:
        return total_prbs, 0
    q_crit = int(round(float(share_crit) * total_prbs))
    q_crit = max(0, min(total_prbs, q_crit))
    return q_crit, total_prbs - q_crit

# ========================= TRAIN STORE (LAZY LOAD) =========================
class TrainStore:
    def __init__(self, folder: Path):
        self.folder = folder
        self.cache: Dict[int, pd.DataFrame] = {}
        self.index: Dict[int, Path] = self._build_index(folder)

    @staticmethod
    def _build_index(folder: Path) -> Dict[int, Path]:
        idx: Dict[int, Path] = {}
        if not folder.exists():
            raise FileNotFoundError(f"Missing trains folder: {folder}")
        for fp in sorted(folder.glob("*.csv")):
            m = re.search(r"train_(\d+)", fp.stem.lower())
            if not m:
                continue
            tid = int(m.group(1))
            idx[tid] = fp
        if not idx:
            raise FileNotFoundError(f"No train_<id>.csv found in {folder}")
        return idx

    def get_row(self, train_id: int, step: int) -> Optional[pd.Series]:
        if train_id not in self.index:
            return None
        if train_id not in self.cache:
            df = pd.read_csv(self.index[train_id])
            if STEP_COL not in df.columns:
                return None
            df[STEP_COL] = pd.to_numeric(df[STEP_COL], errors="coerce")
            df = df.dropna(subset=[STEP_COL]).copy()
            df[STEP_COL] = df[STEP_COL].astype(int)
            df = df.set_index(STEP_COL, drop=False).sort_index()

            for c in CQI_SB_COLS:
                if c not in df.columns:
                    df[c] = np.nan

            self.cache[train_id] = df

        df = self.cache[train_id]
        if step not in df.index:
            return None
        row = df.loc[step]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return row

# ========================= LOAD FINAL DATASET =========================
def load_presence_from_final_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing final dataset: {path}")

    df = pd.read_csv(path)
    required = {TIME_COL, CI_COL, TRAIN_IDS_COL}
    if not required.issubset(set(df.columns)):
        raise ValueError(f"Final dataset must contain columns: {sorted(required)}")

    df[TIME_COL] = pd.to_numeric(df[TIME_COL], errors="coerce")
    df[CI_COL] = pd.to_numeric(df[CI_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL, CI_COL]).copy()
    df[CI_COL] = df[CI_COL].astype(int)

    df[TRAIN_IDS_COL] = df[TRAIN_IDS_COL].apply(parse_train_ids)

    df = df[df[TIME_COL] >= float(START_TIME_S)].copy()
    if END_TIME_S is not None:
        df = df[df[TIME_COL] <= float(END_TIME_S)].copy()
    if END_STEP is not None:
        df = df[df[TIME_COL] <= float(END_STEP) * EPISODE_S].copy()

    df[STEP_COL] = (df[TIME_COL] / EPISODE_S).round().astype(int)
    df = df.sort_values([STEP_COL, CI_COL], kind="stable").reset_index(drop=True)
    return df

# ========================= FORECASTER (load saved STGFiLMFormer) =========================
# The model must have been saved like:
# torch.save({"state_dict": best_state, "config": {...}}, path)

class EventFiLM(nn.Module):
    def __init__(self, event_dim, hidden_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(event_dim, 2 * hidden_dim),
            nn.ReLU(),
            nn.Linear(2 * hidden_dim, 2 * hidden_dim),
        )

    def gamma_beta(self, e: torch.Tensor):
        gb = self.mlp(e)
        gamma, beta = gb.chunk(2, dim=-1)
        return gamma, beta

    @staticmethod
    def apply_film(x: torch.Tensor, gamma: torch.Tensor, beta: torch.Tensor):
        while gamma.dim() < x.dim():
            gamma = gamma.unsqueeze(1)
            beta  = beta.unsqueeze(1)
        return gamma * x + beta

class FiLMLinearSTAttention(nn.Module):
    def __init__(self, dim, event_dim, dropout=0.0):
        super().__init__()
        self.wq = nn.Linear(dim, dim, bias=False)
        self.wk = nn.Linear(dim, dim, bias=False)
        self.wv = nn.Linear(dim, dim, bias=False)
        self.drop = nn.Dropout(dropout)
        self.film_qkv = EventFiLM(event_dim, dim)

    def forward(self, x, e, mode: str):
        B, T, N, C = x.shape
        if mode == "spatial":
            xt = x.reshape(B * T, N, C)
            e_bt = e.unsqueeze(1).repeat(1, T, 1, 1).reshape(B * T, N, -1)
            Q0 = self.wq(xt); K0 = self.wk(xt); V0 = self.wv(xt)
            gamma, beta = self.film_qkv.gamma_beta(e_bt)
            Q = EventFiLM.apply_film(Q0, gamma, beta)
            K = EventFiLM.apply_film(K0, gamma, beta)
            V = EventFiLM.apply_film(V0, gamma, beta)
            Q = Q / (N ** 0.5)
            K = K / (N ** 0.5)
            out = (Q @ (K.transpose(1, 2) @ V)).reshape(B, T, N, C)
            return self.drop(out)

        if mode == "temporal":
            xn = x.permute(0, 2, 1, 3).contiguous().reshape(B * N, T, C)
            e_bn = e.reshape(B * N, -1)
            Q0 = self.wq(xn); K0 = self.wk(xn); V0 = self.wv(xn)
            gamma, beta = self.film_qkv.gamma_beta(e_bn)
            Q = EventFiLM.apply_film(Q0, gamma, beta)
            K = EventFiLM.apply_film(K0, gamma, beta)
            V = EventFiLM.apply_film(V0, gamma, beta)
            Q = Q / (T ** 0.5)
            K = K / (T ** 0.5)
            out = (Q @ (K.transpose(1, 2) @ V)).reshape(B, N, T, C).permute(0, 2, 1, 3).contiguous()
            return self.drop(out)

        raise ValueError("mode must be 'spatial' or 'temporal'")

class STGFiLMBlock(nn.Module):
    def __init__(self, dim, event_dim, dropout=0.1):
        super().__init__()
        self.film_x = EventFiLM(event_dim, dim)
        self.att = FiLMLinearSTAttention(dim, event_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * dim, dim),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x, e):
        gamma, beta = self.film_x.gamma_beta(e)
        x = EventFiLM.apply_film(x, gamma, beta)
        a = self.att(x, e, "spatial") + self.att(x, e, "temporal")
        x = self.norm1(x + a)
        x = self.norm2(x + self.ff(x))
        return x

class STGFiLMFormer(nn.Module):
    def __init__(self, in_dim, out_dim, horizon, history, idx_ces, idx_estop,
                 dim=64, K=3, dropout=0.1, event_dim=2):
        super().__init__()
        self.horizon = horizon
        self.out_dim = out_dim
        self.dim = dim
        self.K = K
        self.history = history
        self.idx_ces = idx_ces
        self.idx_estop = idx_estop

        self.in_proj = nn.Linear(in_dim, dim)
        self.time_embed = nn.Parameter(torch.zeros(1, history, 1, dim))
        nn.init.normal_(self.time_embed, std=0.02)
        self.block = STGFiLMBlock(dim, event_dim=event_dim, dropout=dropout)
        self.gate = nn.Linear(dim, dim)
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, horizon * out_dim))

    def build_event_vector(self, x_hist):
        ces = x_hist[..., self.idx_ces].amax(dim=1)
        est = x_hist[..., self.idx_estop].amax(dim=1)
        return torch.stack([ces, est], dim=-1)

    def graph_propagate(self, x, A_hat):
        outs = [x]
        xk = x
        for _ in range(self.K):
            xk = torch.matmul(A_hat, xk)
            outs.append(xk)
        return outs

    def forward(self, x_hist, A_hat):
        B, T, N, F = x_hist.shape
        e = self.build_event_vector(x_hist)
        x = self.in_proj(x_hist) + self.time_embed[:, :T]
        props = self.graph_propagate(x, A_hat)
        h = self.block(props[0], e)
        for k in range(1, len(props)):
            hk = self.block(props[k], e)
            g = torch.sigmoid(self.gate(h))
            h = h + g * hk
        h_last = h[:, -1]
        y = self.head(h_last).view(B, N, self.horizon, self.out_dim).permute(0, 2, 1, 3).contiguous()
        return y

def build_ci_list_and_Ahat(cell_gnb_path: Path):
    rel = pd.read_csv(cell_gnb_path)
    rel.columns = [c.strip() for c in rel.columns]
    rel["CI"] = pd.to_numeric(rel["CI"], errors="coerce").astype("Int64")
    rel["gnb"] = pd.to_numeric(rel["gnb"], errors="coerce").astype("Int64")
    rel = rel.dropna(subset=["CI", "gnb"]).copy()

    ci_list = sorted(rel["CI"].astype(int).unique().tolist())
    N = len(ci_list)
    ci_to_idx = {ci: i for i, ci in enumerate(ci_list)}

    g_to_cells = rel.groupby("gnb")["CI"].apply(lambda s: sorted(s.astype(int).tolist())).to_dict()
    gnb_ids = sorted(g_to_cells.keys())

    A = np.zeros((N, N), dtype=np.float32)

    def add_undirected(u_ci, v_ci):
        if u_ci in ci_to_idx and v_ci in ci_to_idx:
            u = ci_to_idx[u_ci]
            v = ci_to_idx[v_ci]
            A[u, v] = 1.0
            A[v, u] = 1.0

    for _, cells in g_to_cells.items():
        for i in range(len(cells)):
            for j in range(i + 1, len(cells)):
                add_undirected(cells[i], cells[j])

    for g in gnb_ids:
        if (g + 1) in g_to_cells:
            for c1 in g_to_cells[g]:
                for c2 in g_to_cells[g + 1]:
                    add_undirected(c1, c2)

    A = A + np.eye(N, dtype=np.float32)
    D = np.sum(A, axis=1)
    D_inv_sqrt = np.power(D, -0.5, where=(D > 0))
    D_inv_sqrt[np.isinf(D_inv_sqrt)] = 0.0
    A_hat = (D_inv_sqrt[:, None] * A) * D_inv_sqrt[None, :]
    return ci_list, torch.tensor(A_hat, dtype=torch.float32, device=DEVICE)

class STGFiLMForecaster:
    """
    Uses the SAME FEAT_COLS/TARGET_COLS/log1p config saved in ckpt["config"].
    Predicts t+1 totals across all cells for critical/performance DL/UL.
    """
    def __init__(self, ckpt_path: Path, dataset_csv: Path, cell_gnb_path: Path):
        ckpt = torch.load(ckpt_path, map_location="cpu")
        cfg = ckpt["config"]

        self.history = int(cfg["HISTORY"])
        self.horizon = int(cfg["HORIZON"])
        self.feat_cols = list(cfg["FEAT_COLS"])
        self.target_cols = list(cfg["TARGET_COLS"])
        self.use_log1p = bool(cfg["USE_LOG1P"])

        idx_ces = self.feat_cols.index("CES_count")
        idx_estop = self.feat_cols.index("EStop_count")

        df = pd.read_csv(dataset_csv)
        df.columns = [c.strip() for c in df.columns]
        df["time_s"] = pd.to_numeric(df["time_s"], errors="coerce")
        df["CI"] = pd.to_numeric(df["CI"], errors="coerce").astype("Int64")
        df = df.dropna(subset=["time_s", "CI"]).copy()
        df["CI"] = df["CI"].astype(int)
        df = df.sort_values(["time_s", "CI"], kind="stable").reset_index(drop=True)

        self.ci_list, self.A_hat = build_ci_list_and_Ahat(cell_gnb_path)
        df = df[df["CI"].isin(self.ci_list)].copy()

        time_vals = np.array(sorted(df["time_s"].unique()))
        self.time_vals = time_vals
        self.time_to_idx = {float(t): i for i, t in enumerate(time_vals)}

        def pivot(col):
            p = df.pivot(index="time_s", columns="CI", values=col).reindex(index=time_vals, columns=self.ci_list)
            return p.to_numpy()

        X_np = np.stack([pivot(c) for c in self.feat_cols], axis=-1)  # [T,N,F]
        X_np = np.nan_to_num(X_np, nan=0.0)

        if self.use_log1p:
            log_idxs = list(range(2, len(self.feat_cols)))  # counts+traffic
            X_np[..., log_idxs] = np.log1p(np.maximum(X_np[..., log_idxs], 0.0))

        self.X = torch.tensor(X_np, dtype=torch.float32)

        self.model = STGFiLMFormer(
            in_dim=len(self.feat_cols),
            out_dim=len(self.target_cols),
            horizon=self.horizon,
            history=self.history,
            idx_ces=idx_ces,
            idx_estop=idx_estop,
            dim=64, K=3, dropout=0.1, event_dim=2
        ).to(DEVICE)
        self.model.load_state_dict(ckpt["state_dict"], strict=True)
        self.model.eval()

    @torch.no_grad()
    def predict_tplus1_totals(self, time_s: float) -> Dict[str, float]:
        t = self.time_to_idx.get(float(time_s), None)
        if t is None:
            return {k: 0.0 for k in self.target_cols}
        if t < self.history - 1:
            return {k: 0.0 for k in self.target_cols}

        x_hist = self.X[t - self.history + 1 : t + 1].unsqueeze(0).to(DEVICE)  # [1,H,N,F]
        pred = self.model(x_hist, self.A_hat)  # [1,HOR,N,O]
        pred_t1 = pred[0, 0]                   # [N,O]

        if self.use_log1p:
            pred_t1 = torch.clamp(torch.expm1(pred_t1), min=0.0)

        totals = pred_t1.detach().cpu().numpy().sum(axis=0)
        out = {self.target_cols[i]: float(totals[i]) for i in range(len(self.target_cols))}
        return out

    def predict_tplus1_per_cell(self, time_s: float) -> Dict[int, Dict[str, float]]:
        t = self.time_to_idx.get(float(time_s), None)
        if t is None:
            return {}
        if t < self.history - 1:
            return {}

        x_hist = self.X[t - self.history + 1 : t + 1].unsqueeze(0).to(DEVICE)  # [1,H,N,F]
        pred = self.model(x_hist, self.A_hat)  # [1,HOR,N,O]
        pred_t1 = pred[0, 0]                   # [N,O]

        if self.use_log1p:
            pred_t1 = torch.clamp(torch.expm1(pred_t1), min=0.0)

        arr = pred_t1.detach().cpu().numpy()  # [N,O]
        out: Dict[int, Dict[str, float]] = {}
        for i, ci in enumerate(self.ci_list):
            out[int(ci)] = {self.target_cols[j]: float(arr[i, j]) for j in range(len(self.target_cols))}
        return out

# ========================= SIMULATOR STATE (step-by-step) =========================
class SimulatorState:
    """
    Stateful simulator so RL can step episode-by-episode.
    Keeps the SAME logic as v4 for serving and puncturing.
    """
    def __init__(self, final_df: pd.DataFrame, train_store: TrainStore, forecaster: Optional[STGMFiLMForecaster]):
        self.final_df = final_df
        self.train_store = train_store
        self.forecaster = forecaster

        self.step_to_ci_trains: Dict[int, Dict[int, List[int]]] = {}
        self.step_to_ces_count: Dict[int, float] = {}
        for step, g in final_df.groupby(STEP_COL, sort=True):
            ci_map: Dict[int, List[int]] = {}
            for _, r in g.iterrows():
                ci = int(r[CI_COL])
                tids = r[TRAIN_IDS_COL] if isinstance(r[TRAIN_IDS_COL], list) else []
                if tids:
                    ci_map[ci] = tids
            self.step_to_ci_trains[int(step)] = ci_map
            if "CES_count" in g.columns:
                self.step_to_ces_count[int(step)] = float(pd.to_numeric(g["CES_count"], errors="coerce").fillna(0.0).sum())
            else:
                self.step_to_ces_count[int(step)] = 0.0

        self.steps = sorted(self.step_to_ci_trains.keys())
        if not self.steps:
            raise ValueError("No steps to simulate after filtering. Check END_TIME_S / END_STEP.")

        # stable cell order for RL (use forecaster list if available)
        if self.forecaster is not None and hasattr(self.forecaster, "ci_list"):
            self.ci_list = [int(c) for c in self.forecaster.ci_list]
        else:
            self.ci_list = sorted({int(c) for c in final_df[CI_COL].dropna().unique().tolist()})

        # obs: per cell [4 current demand + 4 prediction + 2 critical job counts] + global CES
        self.obs_dim = 10 * len(self.ci_list) + 1

        # runtime state
        self.job_seq: Dict[Tuple[int, str, str], int] = {}
        self.active_jobs: Dict[Tuple[int, str, str, int], Job] = {}
        self.completed_jobs: List[Job] = []
        self.dropped_jobs: List[Job] = []
        self.episode_rows: List[dict] = []
        self.ptr = 0
        self.next_stream_arrival_time_s: Dict[Tuple[int, str, str], float] = {}

        # prepared context for current episode
        self._prepared: Optional[dict] = None
        self.prev_pred_ci_link: Dict[Tuple[int, str], Tuple[float, float]] = {}
        self.obs_ema_ci_link: Dict[Tuple[int, str], float] = {}
        self.prev_crit_jobs_ci_link: Dict[Tuple[int, str], int] = {}

    def reset(self):
        self.job_seq = {}
        self.active_jobs = {}
        self.completed_jobs = []
        self.dropped_jobs = []
        self.episode_rows = []
        self.ptr = 0
        self._prepared = None
        self.prev_pred_ci_link = {}
        self.obs_ema_ci_link = {}
        self.prev_crit_jobs_ci_link = {}
        self.next_stream_arrival_time_s = {}

    def done(self) -> bool:
        return self.ptr >= len(self.steps)

    def current_step(self) -> int:
        return self.steps[self.ptr]

    def _jobs_in_ci(self, ci_to_trains: Dict[int, List[int]], train_rows: Dict[int, pd.Series],
                    ci: int, link: str, slice_name: str, now_s: Optional[float] = None) -> List[Job]:
        tids = set(ci_to_trains.get(ci, []))
        out: List[Job] = []
        for jb in self.active_jobs.values():
            if jb.link != link:
                continue
            if jb.slice_name != slice_name:
                continue
            if jb.train_id in tids and jb.remaining_bits > 0:
                if now_s is not None and jb.arrival_time_s > (now_s + 1e-12):
                    continue
                if jb.train_id in train_rows:
                    out.append(jb)
        return out

    def _remaining_demand_bits(self, ci_to_trains: Dict[int, List[int]], train_rows: Dict[int, pd.Series],
                              ci: int, link: str, slice_name: str, now_s: Optional[float] = None) -> float:
        return float(sum(jb.remaining_bits for jb in self._jobs_in_ci(
            ci_to_trains, train_rows, ci, link, slice_name, now_s=now_s
        )))

    def _drop_expired_critical_jobs(self, as_of_time_s: float) -> int:
        """
        Strict Delay-GBR behavior for critical traffic:
        critical packets older than their app PDB are dropped and not carried forward.
        """
        drop_ids: List[Tuple[int, str, str, int]] = []
        for job_id, jb in self.active_jobs.items():
            if jb.slice_name != "critical":
                continue
            if jb.remaining_bits <= 0:
                continue
            app_cfg = APP_PACKET_PROFILE.get(jb.app_name, {})
            pdb_ms = float(app_cfg.get("pdb_ms", np.nan))
            if not np.isfinite(pdb_ms):
                continue
            pdb_s = max(0.0, pdb_ms / 1000.0)
            if (as_of_time_s - jb.arrival_time_s) > (pdb_s + 1e-12):
                jb.dropped_time_s = as_of_time_s
                self.dropped_jobs.append(jb)
                drop_ids.append(job_id)

        for job_id in drop_ids:
            self.active_jobs.pop(job_id, None)
        return len(drop_ids)

    def _build_arrival_times_for_episode(
        self,
        stream_key: Tuple[int, str, str],
        episode_start_s: float,
        episode_end_s: float,
        packet_interval_ms: float,
    ) -> List[float]:
        packet_interval_s = max(float(packet_interval_ms) / 1000.0, MINISLOT_S)
        next_arrival_time_s = self.next_stream_arrival_time_s.get(stream_key, episode_start_s)

        if next_arrival_time_s < episode_start_s:
            skipped_intervals = int((episode_start_s - next_arrival_time_s) // packet_interval_s)
            next_arrival_time_s += skipped_intervals * packet_interval_s
            while next_arrival_time_s < episode_start_s:
                next_arrival_time_s += packet_interval_s

        arrival_times_s: List[float] = []
        while next_arrival_time_s < episode_end_s:
            arrival_times_s.append(float(next_arrival_time_s))
            next_arrival_time_s += packet_interval_s

        self.next_stream_arrival_time_s[stream_key] = float(next_arrival_time_s)
        if not arrival_times_s:
            arrival_times_s = [episode_start_s]
        return arrival_times_s

    def _split_episode_volume_into_jobs(
        self,
        episode_bits_expected: float,
        payload_bits: float,
    ) -> List[float]:
        episode_bits_expected = max(float(episode_bits_expected), 0.0)
        payload_bits = max(float(payload_bits), 1.0)
        if episode_bits_expected <= 0:
            return []

        full_payload_job_count = int(episode_bits_expected // payload_bits)
        remaining_bits = episode_bits_expected - (full_payload_job_count * payload_bits)

        job_sizes_bits: List[float] = [payload_bits] * full_payload_job_count
        if remaining_bits > 1e-9:
            job_sizes_bits.append(float(remaining_bits))
        return job_sizes_bits

    def prepare_episode(self) -> np.ndarray:
        """
        Prepare arrivals and build observation BEFORE scheduling.
        Returns obs vector for RL.
        """
        if self.done():
            return np.zeros(self.obs_dim, dtype=np.float32)

        step = self.current_step()
        episode_start_s = step * EPISODE_S
        dropped_critical_jobs = self._drop_expired_critical_jobs(episode_start_s)
        ci_to_trains = self.step_to_ci_trains.get(step, {})
        ces_count_episode = float(self.step_to_ces_count.get(step, 0.0))

        active_train_ids = sorted({tid for tids in ci_to_trains.values() for tid in tids})
        train_rows: Dict[int, pd.Series] = {}
        for tid in active_train_ids:
            row = self.train_store.get_row(tid, step)
            if row is None:
                continue
            train_rows[tid] = row

        # arrivals split into intra-episode jobs using per-app payload and interval profiles
        demand_bits_arrivals = _init_slice_link_dict(0.0)
        episode_job_meta: List[dict] = []
        episode_end_s = episode_start_s + EPISODE_S
        for tid, row in train_rows.items():
            for app_name, meta in APP_DEFS.items():
                slice_name = meta["slice"]
                packet_profile = APP_PACKET_PROFILE.get(app_name, {})
                payload_bits = float(packet_profile.get("payload_bits", 1600.0))
                packet_interval_ms = float(packet_profile.get("interval_ms", 20.0))

                dl_kb = pd.to_numeric(row.get(meta["dl"], 0.0), errors="coerce")
                episode_bits_expected_dl = float(dl_kb) * KB_TO_BITS if (not pd.isna(dl_kb)) else 0.0
                if episode_bits_expected_dl > 0:
                    stream_key_dl = (tid, app_name, "DL")
                    arrival_times_dl = self._build_arrival_times_for_episode(
                        stream_key_dl,
                        episode_start_s,
                        episode_end_s,
                        packet_interval_ms,
                    )
                    job_sizes_bits_dl = self._split_episode_volume_into_jobs(
                        episode_bits_expected_dl,
                        payload_bits,
                    )

                    job_count_dl = len(job_sizes_bits_dl)
                    arrival_count_dl = len(arrival_times_dl)
                    for job_index, job_bits in enumerate(job_sizes_bits_dl):
                        arrival_index = int((job_index * arrival_count_dl) // max(job_count_dl, 1))
                        arrival_index = min(arrival_index, arrival_count_dl - 1)
                        arrival_time_s = arrival_times_dl[arrival_index]

                        seq = self.job_seq.get(stream_key_dl, 0) + 1
                        self.job_seq[stream_key_dl] = seq
                        job_id = (tid, app_name, "DL", seq)
                        self.active_jobs[job_id] = Job(
                            job_id=job_id,
                            train_id=tid,
                            app_name=app_name,
                            link="DL",
                            slice_name=slice_name,
                            arrival_time_s=arrival_time_s,
                            remaining_bits=job_bits,
                            total_bits=job_bits,
                        )
                        demand_bits_arrivals[(slice_name, "DL")] += job_bits
                        episode_job_meta.append({
                            "job_id": job_id,
                            "slice": slice_name,
                            "link": "DL",
                            "app": app_name,
                            "demand_bits": job_bits,
                        })

                ul_kb = pd.to_numeric(row.get(meta["ul"], 0.0), errors="coerce")
                episode_bits_expected_ul = float(ul_kb) * KB_TO_BITS if (not pd.isna(ul_kb)) else 0.0
                if episode_bits_expected_ul > 0:
                    stream_key_ul = (tid, app_name, "UL")
                    arrival_times_ul = self._build_arrival_times_for_episode(
                        stream_key_ul,
                        episode_start_s,
                        episode_end_s,
                        packet_interval_ms,
                    )
                    job_sizes_bits_ul = self._split_episode_volume_into_jobs(
                        episode_bits_expected_ul,
                        payload_bits,
                    )

                    job_count_ul = len(job_sizes_bits_ul)
                    arrival_count_ul = len(arrival_times_ul)
                    for job_index, job_bits in enumerate(job_sizes_bits_ul):
                        arrival_index = int((job_index * arrival_count_ul) // max(job_count_ul, 1))
                        arrival_index = min(arrival_index, arrival_count_ul - 1)
                        arrival_time_s = arrival_times_ul[arrival_index]

                        seq = self.job_seq.get(stream_key_ul, 0) + 1
                        self.job_seq[stream_key_ul] = seq
                        job_id = (tid, app_name, "UL", seq)
                        self.active_jobs[job_id] = Job(
                            job_id=job_id,
                            train_id=tid,
                            app_name=app_name,
                            link="UL",
                            slice_name=slice_name,
                            arrival_time_s=arrival_time_s,
                            remaining_bits=job_bits,
                            total_bits=job_bits,
                        )
                        demand_bits_arrivals[(slice_name, "UL")] += job_bits
                        episode_job_meta.append({
                            "job_id": job_id,
                            "slice": slice_name,
                            "link": "UL",
                            "app": app_name,
                            "demand_bits": job_bits,
                        })

        # total demand at episode start (backlog + arrivals) per slice/link
        demand_bits_total_start = _init_slice_link_dict(0.0)
        for jb in self.active_jobs.values():
            if jb.remaining_bits > 0:
                demand_bits_total_start[(jb.slice_name, jb.link)] += jb.remaining_bits
        pred_per_cell = {}
        if self.forecaster is not None and USE_FORECAST:
            pred_pc = self.forecaster.predict_tplus1_per_cell(episode_start_s)
            for ci, d in pred_pc.items():
                pred_per_cell[int(ci)] = {
                    "critical_DL": float(d.get("critical_DL_sum", 0.0)) * KB_TO_BITS,
                    "performance_DL": float(d.get("perform_DL_sum", 0.0)) * KB_TO_BITS,
                    "critical_UL": float(d.get("critical_UL_sum", 0.0)) * KB_TO_BITS,
                    "performance_UL": float(d.get("perform_UL_sum", 0.0)) * KB_TO_BITS,
                }

        # current demand/prediction/job-count per cell + global CES for DRL observation
        obs_vals: List[float] = []
        for ci in self.ci_list:
            cur_crit_dl = self._remaining_demand_bits(ci_to_trains, train_rows, ci, "DL", "critical")
            cur_perf_dl = self._remaining_demand_bits(ci_to_trains, train_rows, ci, "DL", "performance")
            cur_crit_ul = self._remaining_demand_bits(ci_to_trains, train_rows, ci, "UL", "critical")
            cur_perf_ul = self._remaining_demand_bits(ci_to_trains, train_rows, ci, "UL", "performance")

            pred_ci = pred_per_cell.get(ci, {}) if isinstance(pred_per_cell, dict) else {}
            pred_crit_dl = float(pred_ci.get("critical_DL", 0.0))
            pred_perf_dl = float(pred_ci.get("performance_DL", 0.0))
            pred_crit_ul = float(pred_ci.get("critical_UL", 0.0))
            pred_perf_ul = float(pred_ci.get("performance_UL", 0.0))

            ncrit_dl = float(len(self._jobs_in_ci(ci_to_trains, train_rows, ci, "DL", "critical")))
            ncrit_ul = float(len(self._jobs_in_ci(ci_to_trains, train_rows, ci, "UL", "critical")))

            obs_vals.extend([
                cur_crit_dl, cur_perf_dl,
                cur_crit_ul, cur_perf_ul,
                pred_crit_dl, pred_perf_dl,
                pred_crit_ul, pred_perf_ul,
                ncrit_dl, ncrit_ul,
            ])

        obs_vals.append(float(ces_count_episode))
        obs = np.log1p(np.maximum(np.array(obs_vals, dtype=np.float32), 0.0))

        # store prepared context for scheduling
        self._prepared = {
            "step": step,
            "episode_start_s": episode_start_s,
            "ci_to_trains": ci_to_trains,
            "active_train_ids": active_train_ids,
            "train_rows": train_rows,
            "demand_bits_arrivals": demand_bits_arrivals,
            "pred_per_cell": pred_per_cell,
            "demand_bits_total_start": demand_bits_total_start,
            "episode_job_meta": episode_job_meta,
            "ces_count_episode": ces_count_episode,
            "dropped_critical_jobs": dropped_critical_jobs,
        }
        return obs

    def run_prepared_episode(self, mode: str, action: Optional[np.ndarray] = None) -> Tuple[dict, float]:
        """
        mode:
          - "naive" => quotas computed from demand proportions per cell/link
          - "proreactive" => proactive quota + confidence-gated reactive fallback
        action:
          array([share_crit_dl, share_crit_ul]) in [0,1]
        Returns:
          episode_metrics_row, reward (for RL training)
        """
        assert self._prepared is not None, "Call prepare_episode() first"
        ctx = self._prepared

        t_start = time.perf_counter()
        e_start_uj = _read_rapl_uj()

        step = ctx["step"]
        episode_start_s = ctx["episode_start_s"]
        ci_to_trains = ctx["ci_to_trains"]
        train_rows: Dict[int, pd.Series] = ctx["train_rows"]

        demand_bits_arrivals = ctx["demand_bits_arrivals"]
        demand_bits_total_start = ctx.get("demand_bits_total_start", demand_bits_arrivals)
        pred_per_cell = ctx.get("pred_per_cell", {})
        episode_job_meta = ctx.get("episode_job_meta", [])
        ces_count_episode = float(ctx.get("ces_count_episode", 0.0))
        dropped_critical_jobs = int(ctx.get("dropped_critical_jobs", 0))
        emergency_mode = 1 if ces_count_episode > 0 else 0
        alloc_bits_episode_job = {m["job_id"]: 0.0 for m in episode_job_meta}

        drl_share_dl = 0.5
        drl_share_ul = 0.5
        drl_gate_dl = 0.5
        drl_gate_ul = 0.5
        if mode == "proreactive_drl":
            if action is None:
                raise ValueError("proreactive_drl requires action=[share_dl, share_ul, gate_dl, gate_ul]")
            act = np.clip(np.asarray(action, dtype=np.float32).ravel(), 0.0, 1.0)
            if len(act) < 4:
                raise ValueError("proreactive_drl action must have 4 values")
            drl_share_dl, drl_share_ul, drl_gate_dl, drl_gate_ul = [float(v) for v in act[:4]]

        served_bits = _init_slice_link_dict(0.0)
        punctured_prbs = {"DL": 0, "UL": 0}

        # gamma per train
        gamma_dl: Dict[int, np.ndarray] = {}
        gamma_ul: Dict[int, np.ndarray] = {}
        for tid, row in train_rows.items():
            gamma_dl[tid] = build_gamma_bits_per_prb(row, K_PRB_DL)
            gamma_ul[tid] = build_gamma_bits_per_prb(row, K_PRB_UL)

        def _best_gamma_job(jobs, gamma_map, prb, k_prb):
            best_jb = None
            best_bits = 0.0
            zeros = np.zeros(k_prb)
            for jb in jobs:
                bits = float(gamma_map.get(jb.train_id, zeros)[prb])
                if bits > best_bits:
                    best_bits = bits
                    best_jb = jb
            return best_jb, best_bits


        def _pick_job(jobs: List[Job], mode_name: str, slice_name: str) -> Optional[Job]:
            if not jobs:
                return None
            return pick_job_fcfs(jobs)

        # quotas per cell/link
        quotas: Dict[Tuple[int, str], Tuple[int, int]] = {}
        quotas_naive: Dict[Tuple[int, str], Tuple[int, int]] = {}
        quotas_proactive: Dict[Tuple[int, str], Tuple[int, int]] = {}
        quotas_drl: Dict[Tuple[int, str], Tuple[int, int]] = {}

        share_by_ci: Dict[int, Tuple[float, float]] = {}


        for ci in ci_to_trains.keys():
            d_crit_dl = self._remaining_demand_bits(ci_to_trains, train_rows, ci, "DL", "critical")
            d_perf_dl = self._remaining_demand_bits(ci_to_trains, train_rows, ci, "DL", "performance")
            d_crit_ul = self._remaining_demand_bits(ci_to_trains, train_rows, ci, "UL", "critical")
            d_perf_ul = self._remaining_demand_bits(ci_to_trains, train_rows, ci, "UL", "performance")

            qcdl_naive, _ = compute_quota_prbs(d_crit_dl, d_perf_dl, K_PRB_DL)
            qcul_naive, _ = compute_quota_prbs(d_crit_ul, d_perf_ul, K_PRB_UL)
            quotas_naive[(ci, "DL")] = (qcdl_naive, K_PRB_DL - qcdl_naive)
            quotas_naive[(ci, "UL")] = (qcul_naive, K_PRB_UL - qcul_naive)

            pred_ci = pred_per_cell.get(ci, {}) if isinstance(pred_per_cell, dict) else {}
            pd_crit_dl = float(pred_ci.get("critical_DL", 0.0))
            pd_perf_dl = float(pred_ci.get("performance_DL", 0.0))
            pd_crit_ul = float(pred_ci.get("critical_UL", 0.0))
            pd_perf_ul = float(pred_ci.get("performance_UL", 0.0))

            d_crit_dl_eff = d_crit_dl + PROREACTIVE_PRED_WEIGHT * pd_crit_dl
            d_perf_dl_eff = d_perf_dl + PROREACTIVE_PRED_WEIGHT * pd_perf_dl
            d_crit_ul_eff = d_crit_ul + PROREACTIVE_PRED_WEIGHT * pd_crit_ul
            d_perf_ul_eff = d_perf_ul + PROREACTIVE_PRED_WEIGHT * pd_perf_ul

            qcdl_pro, _ = compute_quota_prbs(d_crit_dl_eff, d_perf_dl_eff, K_PRB_DL)
            qcul_pro, _ = compute_quota_prbs(d_crit_ul_eff, d_perf_ul_eff, K_PRB_UL)
            quotas_proactive[(ci, "DL")] = (qcdl_pro, K_PRB_DL - qcdl_pro)
            quotas_proactive[(ci, "UL")] = (qcul_pro, K_PRB_UL - qcul_pro)

            # DRL quota shares (guardrails avoid wasting PRBs on empty critical/performance)
            qcdl_drl, _ = shares_to_quota(drl_share_dl, K_PRB_DL, d_crit_dl, d_perf_dl)
            qcul_drl, _ = shares_to_quota(drl_share_ul, K_PRB_UL, d_crit_ul, d_perf_ul)
            quotas_drl[(ci, "DL")] = (qcdl_drl, K_PRB_DL - qcdl_drl)
            quotas_drl[(ci, "UL")] = (qcul_drl, K_PRB_UL - qcul_drl)

            if mode == "naive":
                quotas[(ci, "DL")] = quotas_naive[(ci, "DL")]
                quotas[(ci, "UL")] = quotas_naive[(ci, "UL")]
            elif mode in ("proreactive", "proactive", "proreactive_drl"):
                quotas[(ci, "DL")] = quotas_proactive[(ci, "DL")]
                quotas[(ci, "UL")] = quotas_proactive[(ci, "UL")]
            else:
                raise ValueError(f"Unknown mode: {mode}")

        # critical backlog per cell/link (updated as we serve)
        crit_backlog_dl = {}
        crit_backlog_ul = {}
        for ci in ci_to_trains.keys():
            crit_backlog_dl[ci] = self._remaining_demand_bits(ci_to_trains, train_rows, ci, "DL", "critical")
            crit_backlog_ul[ci] = self._remaining_demand_bits(ci_to_trains, train_rows, ci, "UL", "critical")

        last_punctured_dl = {ci: None for ci in ci_to_trains.keys()}
        last_punctured_ul = {ci: None for ci in ci_to_trains.keys()}

        # Episode-start simple trigger for pro-reactive fallback:
        # switch if critical backlog jobs > 10 OR previous forecast underestimation <= -10% of real.
        allow_puncture_dl = {ci: False for ci in ci_to_trains.keys()}
        allow_puncture_ul = {ci: False for ci in ci_to_trains.keys()}
        for ci in ci_to_trains.keys():
            for link in ["DL", "UL"]:
                obs_crit = self._remaining_demand_bits(ci_to_trains, train_rows, ci, link, "critical")
                obs_perf = self._remaining_demand_bits(ci_to_trains, train_rows, ci, link, "performance")
                obs_tot = obs_crit + obs_perf

                pred_ci = pred_per_cell.get(ci, {}) if isinstance(pred_per_cell, dict) else {}
                pred_crit = float(pred_ci.get("critical_" + link, 0.0))
                pred_perf = float(pred_ci.get("performance_" + link, 0.0))

                prev_pred_crit, prev_pred_perf = self.prev_pred_ci_link.get((ci, link), (pred_crit, pred_perf))
                prev_pred_tot = prev_pred_crit + prev_pred_perf
                rel_err = (prev_pred_tot - obs_tot) / (obs_tot + 1e-9)

                crit_jobs_now = len(self._jobs_in_ci(
                    ci_to_trains, train_rows, ci, link, "critical", now_s=episode_start_s
                ))
                backlog_trigger = crit_jobs_now > 10
                error_trigger = rel_err <= -0.10

                if mode == "proreactive_drl":
                    backlog_norm = min(float(crit_jobs_now) / 20.0, 1.0)
                    underpred_ratio = max((obs_tot - prev_pred_tot) / (obs_tot + 1e-9), 0.0)
                    ces_norm = min(float(ces_count_episode) / 5.0, 1.0)
                    risk_score = 0.5 * backlog_norm + 0.3 * underpred_ratio + 0.2 * ces_norm
                    gate = drl_gate_dl if link == "DL" else drl_gate_ul
                    do_puncture = risk_score >= gate
                else:
                    do_puncture = backlog_trigger or error_trigger

                if link == "DL":
                    allow_puncture_dl[ci] = do_puncture
                else:
                    allow_puncture_ul[ci] = do_puncture

                self.prev_pred_ci_link[(ci, link)] = (pred_crit, pred_perf)
                self.prev_crit_jobs_ci_link[(ci, link)] = crit_jobs_now

        # Effective per-cell/link behavior for ProReactive:
        # proactive path == same as Proactive, reactive path == same as Reactive.
        effective_mode_dl: Dict[int, str] = {}
        effective_mode_ul: Dict[int, str] = {}
        for ci in ci_to_trains.keys():
            if mode == "naive":
                effective_mode_dl[ci] = "naive"
                effective_mode_ul[ci] = "naive"
            elif mode == "proactive":
                effective_mode_dl[ci] = "proactive"
                effective_mode_ul[ci] = "proactive"
            else:
                effective_mode_dl[ci] = "naive" if allow_puncture_dl.get(ci, False) else "proactive"
                effective_mode_ul[ci] = "naive" if allow_puncture_ul.get(ci, False) else "proactive"

            # make ProReactive exactly proactive or reactive (including quota) per link/cell
            if mode == "proreactive_drl":
                quotas[(ci, "DL")] = quotas_drl[(ci, "DL")] if effective_mode_dl[ci] == "proactive" else quotas_naive[(ci, "DL")]
                quotas[(ci, "UL")] = quotas_drl[(ci, "UL")] if effective_mode_ul[ci] == "proactive" else quotas_naive[(ci, "UL")]
            else:
                quotas[(ci, "DL")] = quotas_proactive[(ci, "DL")] if effective_mode_dl[ci] == "proactive" else quotas_naive[(ci, "DL")]
                quotas[(ci, "UL")] = quotas_proactive[(ci, "UL")] if effective_mode_ul[ci] == "proactive" else quotas_naive[(ci, "UL")]

        # Proactive contiguous quota over episode RB-time grid
        qcrit_cells_reserved = {}
        qcrit_cells_used = {}
        for ci in ci_to_trains.keys():
            qcdl_base, _ = quotas[(ci, "DL")]
            qcul_base, _ = quotas[(ci, "UL")]
            if effective_mode_dl.get(ci) == "proactive":
                qcrit_cells_reserved[(ci, "DL")] = int(qcdl_base * MINISLOTS_PER_EPISODE)
                qcrit_cells_used[(ci, "DL")] = 0
            if effective_mode_ul.get(ci) == "proactive":
                qcrit_cells_reserved[(ci, "UL")] = int(qcul_base * MINISLOTS_PER_EPISODE)
                qcrit_cells_used[(ci, "UL")] = 0

        # minislot loop
        for ms in range(MINISLOTS_PER_EPISODE):
            now_s = episode_start_s + ms * MINISLOT_S

            for ci in ci_to_trains.keys():
                dl_mode = effective_mode_dl.get(ci, "naive")
                ul_mode = effective_mode_ul.get(ci, "naive")
                strict_ul_critical_only = (mode in ("proreactive", "proreactive_drl") and ul_mode == "naive")

                # ---- DL ----
                if dl_mode == "proactive":
                    rem_dl = qcrit_cells_reserved[(ci, "DL")] - qcrit_cells_used[(ci, "DL")]
                    q_crit_dl = int(max(0, min(K_PRB_DL, rem_dl)))
                else:
                    q_crit_dl, _q_perf_dl = quotas[(ci, "DL")]

                # critical quota
                for prb in range(q_crit_dl):
                    crit_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "DL", "critical", now_s=now_s)
                    if dl_mode == "proactive":
                        if not crit_jobs:
                            # give unused critical PRB to performance
                            perf_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "DL", "performance", now_s=now_s)
                            if not perf_jobs:
                                break
                            if dl_mode == "proactive":
                                jb, bits = _best_gamma_job(perf_jobs, gamma_dl, prb, K_PRB_DL)
                                if jb is None or bits <= 0:
                                    continue
                            else:
                                jb = _pick_job(perf_jobs, mode, "performance")
                                if jb is None:
                                    continue
                                bits = float(gamma_dl.get(jb.train_id, np.zeros(K_PRB_DL))[prb])
                                if bits <= 0:
                                    continue
                            serve = bits
                            if serve <= 0:
                                continue
                            if jb.first_served_time_s is None:
                                jb.first_served_time_s = now_s
                            jb.remaining_bits = max(0.0, jb.remaining_bits - serve)
                            served_bits[(jb.slice_name, "DL")] += serve
                            if jb.job_id in alloc_bits_episode_job:
                                alloc_bits_episode_job[jb.job_id] += serve
                            if jb.remaining_bits <= 0 and jb.done_time_s is None:
                                jb.done_time_s = now_s + MINISLOT_S
                                self.completed_jobs.append(jb)
                                self.active_jobs.pop(jb.job_id, None)
                            continue

                        if dl_mode == "proactive":
                            jb, bits = _best_gamma_job(crit_jobs, gamma_dl, prb, K_PRB_DL)
                            if jb is None or bits <= 0:
                                continue
                        else:
                            jb = _pick_job(crit_jobs, mode, "critical")
                            if jb is None:
                                continue
                            bits = float(gamma_dl.get(jb.train_id, np.zeros(K_PRB_DL))[prb])
                            if bits <= 0:
                                continue
                    else:
                        if not crit_jobs:
                            break
                        jb = _pick_job(crit_jobs, mode, "critical")
                        if jb is None:
                            break
                        bits = float(gamma_dl.get(jb.train_id, np.zeros(K_PRB_DL))[prb])
                        if bits <= 0:
                            continue

                    serve = bits
                    if serve <= 0:
                        continue

                    if jb.first_served_time_s is None:
                        jb.first_served_time_s = now_s
                    jb.remaining_bits = max(0.0, jb.remaining_bits - serve)
                    served_bits[(jb.slice_name, "DL")] += serve
                    if jb.job_id in alloc_bits_episode_job:
                        alloc_bits_episode_job[jb.job_id] += serve
                    if jb.slice_name == "critical":
                        crit_backlog_dl[ci] = max(0.0, crit_backlog_dl.get(ci, 0.0) - serve)

                    if jb.remaining_bits <= 0 and jb.done_time_s is None:
                        jb.done_time_s = now_s + MINISLOT_S
                        self.completed_jobs.append(jb)
                        self.active_jobs.pop(jb.job_id, None)

                # ProReactive uses exact proactive branch or exact reactive branch per trigger.
                if dl_mode == "proactive":
                    for prb in range(q_crit_dl, K_PRB_DL):
                        perf_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "DL", "performance", now_s=now_s)
                        if not perf_jobs:
                            continue
                        if dl_mode == "proactive":
                            jb, bits = _best_gamma_job(perf_jobs, gamma_dl, prb, K_PRB_DL)
                            if jb is None or bits <= 0:
                                continue
                        else:
                            jb = _pick_job(perf_jobs, mode, "performance")
                            if jb is None:
                                continue
                            tid = jb.train_id
                            bits = float(gamma_dl.get(tid, np.zeros(K_PRB_DL))[prb])
                        if bits <= 0:
                            continue
                        serve = bits
                        if serve <= 0:
                            continue
                        if jb.first_served_time_s is None:
                            jb.first_served_time_s = now_s
                        jb.remaining_bits = max(0.0, jb.remaining_bits - serve)
                        served_bits[(jb.slice_name, "DL")] += serve
                        if jb.job_id in alloc_bits_episode_job:
                            alloc_bits_episode_job[jb.job_id] += serve
                        if jb.remaining_bits <= 0 and jb.done_time_s is None:
                            jb.done_time_s = now_s + MINISLOT_S
                            self.completed_jobs.append(jb)
                            self.active_jobs.pop(jb.job_id, None)
                else:
                    # knapsack-inspired puncturing: rank performance PRBs by profit/weight
                    crit_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "DL", "critical", now_s=now_s)
                    perf_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "DL", "performance", now_s=now_s)
                    prb_candidates = list(range(q_crit_dl, K_PRB_DL))

                    if perf_jobs:
                        ratios = []
                        for prb in prb_candidates:
                            _, profit = _best_gamma_job(crit_jobs, gamma_dl, prb, K_PRB_DL)
                            _, weight = _best_gamma_job(perf_jobs, gamma_dl, prb, K_PRB_DL)
                            ratio = profit / max(weight, 1e-9)
                            ratios.append((ratio, prb))
                        ratios.sort(reverse=True)
                        prb_order = [p for _, p in ratios]
                        lp = last_punctured_dl.get(ci, None)
                        if lp is not None and len(prb_order) > 1 and prb_order[0] == lp:
                            prb_order[0], prb_order[1] = prb_order[1], prb_order[0]
                    else:
                        prb_order = prb_candidates

                    for prb in prb_order:
                        crit_backlog = crit_backlog_dl.get(ci, 0.0)
                        if crit_backlog > 0 and crit_jobs:
                            jb = _pick_job(crit_jobs, mode, "critical")
                            if jb is None:
                                continue
                            tid = jb.train_id
                            bits = float(gamma_dl.get(tid, np.zeros(K_PRB_DL))[prb])
                            if bits <= 0:
                                continue
                            punctured_prbs["DL"] += 1
                            serve = bits
                            if serve <= 0:
                                continue
                            if jb.first_served_time_s is None:
                                jb.first_served_time_s = now_s
                            jb.remaining_bits = max(0.0, jb.remaining_bits - serve)
                            served_bits[(jb.slice_name, "DL")] += serve
                            if jb.job_id in alloc_bits_episode_job:
                                alloc_bits_episode_job[jb.job_id] += serve
                            crit_backlog_dl[ci] = max(0.0, crit_backlog_dl.get(ci, 0.0) - serve)
                            last_punctured_dl[ci] = prb
                            if jb.remaining_bits <= 0 and jb.done_time_s is None:
                                jb.done_time_s = now_s + MINISLOT_S
                                self.completed_jobs.append(jb)
                                self.active_jobs.pop(jb.job_id, None)
                        else:
                            # serve performance normally
                            if not perf_jobs:
                                continue
                            jb = _pick_job(perf_jobs, mode, "performance")
                            if jb is None:
                                continue
                            tid = jb.train_id
                            bits = float(gamma_dl.get(tid, np.zeros(K_PRB_DL))[prb])
                            if bits <= 0:
                                continue
                            serve = bits
                            if serve <= 0:
                                continue
                            if jb.first_served_time_s is None:
                                jb.first_served_time_s = now_s
                            jb.remaining_bits = max(0.0, jb.remaining_bits - serve)
                            served_bits[(jb.slice_name, "DL")] += serve
                            if jb.job_id in alloc_bits_episode_job:
                                alloc_bits_episode_job[jb.job_id] += serve
                            if jb.remaining_bits <= 0 and jb.done_time_s is None:
                                jb.done_time_s = now_s + MINISLOT_S
                                self.completed_jobs.append(jb)
                                self.active_jobs.pop(jb.job_id, None)

                if dl_mode == "proactive":
                    qcrit_cells_used[(ci, "DL")] += q_crit_dl

# ---- UL ----
                if ul_mode == "proactive":
                    rem_ul = qcrit_cells_reserved[(ci, "UL")] - qcrit_cells_used[(ci, "UL")]
                    q_crit_ul = int(max(0, min(K_PRB_UL, rem_ul)))
                else:
                    q_crit_ul, _q_perf_ul = quotas[(ci, "UL")]
                    if strict_ul_critical_only:
                        q_crit_ul = K_PRB_UL

                for prb in range(q_crit_ul):
                    crit_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "UL", "critical", now_s=now_s)
                    if ul_mode == "proactive":
                        if not crit_jobs:
                            perf_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "UL", "performance", now_s=now_s)
                            if not perf_jobs:
                                break
                            if ul_mode == "proactive":
                                jb, bits = _best_gamma_job(perf_jobs, gamma_ul, prb, K_PRB_UL)
                                if jb is None or bits <= 0:
                                    continue
                            else:
                                jb = _pick_job(perf_jobs, mode, "performance")
                                if jb is None:
                                    continue
                                bits = float(gamma_ul.get(jb.train_id, np.zeros(K_PRB_UL))[prb])
                                if bits <= 0:
                                    continue
                            serve = bits
                            if serve <= 0:
                                continue
                            if jb.first_served_time_s is None:
                                jb.first_served_time_s = now_s
                            jb.remaining_bits = max(0.0, jb.remaining_bits - serve)
                            served_bits[(jb.slice_name, "UL")] += serve
                            if jb.job_id in alloc_bits_episode_job:
                                alloc_bits_episode_job[jb.job_id] += serve
                            if jb.remaining_bits <= 0 and jb.done_time_s is None:
                                jb.done_time_s = now_s + MINISLOT_S
                                self.completed_jobs.append(jb)
                                self.active_jobs.pop(jb.job_id, None)
                            continue

                        if ul_mode == "proactive":
                            jb, bits = _best_gamma_job(crit_jobs, gamma_ul, prb, K_PRB_UL)
                            if jb is None or bits <= 0:
                                continue
                        else:
                            jb = _pick_job(crit_jobs, mode, "critical")
                            if jb is None:
                                continue
                            bits = float(gamma_ul.get(jb.train_id, np.zeros(K_PRB_UL))[prb])
                            if bits <= 0:
                                continue
                    else:
                        if not crit_jobs:
                            if strict_ul_critical_only:
                                perf_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "UL", "performance", now_s=now_s)
                                if not perf_jobs:
                                    break
                                jb = _pick_job(perf_jobs, mode, "performance")
                                if jb is None:
                                    continue
                                bits = float(gamma_ul.get(jb.train_id, np.zeros(K_PRB_UL))[prb])
                                if bits <= 0:
                                    continue
                            else:
                                break
                        else:
                            jb = _pick_job(crit_jobs, mode, "critical")
                            if jb is None:
                                break
                            bits = float(gamma_ul.get(jb.train_id, np.zeros(K_PRB_UL))[prb])
                            if bits <= 0:
                                continue

                    serve = bits
                    if serve <= 0:
                        continue

                    if jb.first_served_time_s is None:
                        jb.first_served_time_s = now_s
                    jb.remaining_bits = max(0.0, jb.remaining_bits - serve)
                    served_bits[(jb.slice_name, "UL")] += serve
                    if jb.job_id in alloc_bits_episode_job:
                        alloc_bits_episode_job[jb.job_id] += serve
                    if jb.slice_name == "critical":
                        crit_backlog_ul[ci] = max(0.0, crit_backlog_ul.get(ci, 0.0) - serve)

                    if jb.remaining_bits <= 0 and jb.done_time_s is None:
                        jb.done_time_s = now_s + MINISLOT_S
                        self.completed_jobs.append(jb)
                        self.active_jobs.pop(jb.job_id, None)
                for prb in range(q_crit_ul, K_PRB_UL):
                    if ul_mode == "proactive":
                        perf_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "UL", "performance", now_s=now_s)
                        if not perf_jobs:
                            continue
                        if ul_mode == "proactive":
                            jb, bits = _best_gamma_job(perf_jobs, gamma_ul, prb, K_PRB_UL)
                            if jb is None or bits <= 0:
                                continue
                        else:
                            jb = _pick_job(perf_jobs, mode, "performance")
                            if jb is None:
                                continue
                            tid = jb.train_id
                            bits = float(gamma_ul.get(tid, np.zeros(K_PRB_UL))[prb])
                        if bits <= 0:
                            continue
                        serve = bits
                        if serve <= 0:
                            continue
                        if jb.first_served_time_s is None:
                            jb.first_served_time_s = now_s
                        jb.remaining_bits = max(0.0, jb.remaining_bits - serve)
                        served_bits[(jb.slice_name, "UL")] += serve
                        if jb.job_id in alloc_bits_episode_job:
                            alloc_bits_episode_job[jb.job_id] += serve
                        if jb.remaining_bits <= 0 and jb.done_time_s is None:
                            jb.done_time_s = now_s + MINISLOT_S
                            self.completed_jobs.append(jb)
                            self.active_jobs.pop(jb.job_id, None)
                    else:
                        # knapsack-inspired puncturing: rank performance PRBs by profit/weight
                        crit_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "UL", "critical", now_s=now_s)
                        perf_jobs = self._jobs_in_ci(ci_to_trains, train_rows, ci, "UL", "performance", now_s=now_s)
                        prb_candidates = list(range(q_crit_ul, K_PRB_UL))

                        if perf_jobs:
                            ratios = []
                            for prb in prb_candidates:
                                _, profit = _best_gamma_job(crit_jobs, gamma_ul, prb, K_PRB_UL)
                                _, weight = _best_gamma_job(perf_jobs, gamma_ul, prb, K_PRB_UL)
                                ratio = profit / max(weight, 1e-9)
                                ratios.append((ratio, prb))
                            ratios.sort(reverse=True)
                            prb_order = [p for _, p in ratios]
                            lp = last_punctured_ul.get(ci, None)
                            if lp is not None and len(prb_order) > 1 and prb_order[0] == lp:
                                prb_order[0], prb_order[1] = prb_order[1], prb_order[0]
                        else:
                            prb_order = prb_candidates

                        for prb in prb_order:
                            crit_backlog = crit_backlog_ul.get(ci, 0.0)
                            if crit_backlog > 0 and crit_jobs:
                                jb = _pick_job(crit_jobs, mode, "critical")
                                if jb is None:
                                    continue
                                tid = jb.train_id
                                bits = float(gamma_ul.get(tid, np.zeros(K_PRB_UL))[prb])
                                if bits <= 0:
                                    continue
                                punctured_prbs["UL"] += 1
                                serve = bits
                                if serve <= 0:
                                    continue
                                if jb.first_served_time_s is None:
                                    jb.first_served_time_s = now_s
                                jb.remaining_bits = max(0.0, jb.remaining_bits - serve)
                                served_bits[(jb.slice_name, "UL")] += serve
                                if jb.job_id in alloc_bits_episode_job:
                                    alloc_bits_episode_job[jb.job_id] += serve
                                crit_backlog_ul[ci] = max(0.0, crit_backlog_ul.get(ci, 0.0) - serve)
                                last_punctured_ul[ci] = prb
                                if jb.remaining_bits <= 0 and jb.done_time_s is None:
                                    jb.done_time_s = now_s + MINISLOT_S
                                    self.completed_jobs.append(jb)
                                    self.active_jobs.pop(jb.job_id, None)
                            else:
                                if not perf_jobs:
                                    continue
                                jb = _pick_job(perf_jobs, mode, "performance")
                                if jb is None:
                                    continue
                                tid = jb.train_id
                                bits = float(gamma_ul.get(tid, np.zeros(K_PRB_UL))[prb])
                                if bits <= 0:
                                    continue
                                serve = bits
                                if serve <= 0:
                                    continue
                                if jb.first_served_time_s is None:
                                    jb.first_served_time_s = now_s
                                jb.remaining_bits = max(0.0, jb.remaining_bits - serve)
                                served_bits[(jb.slice_name, "UL")] += serve
                                if jb.job_id in alloc_bits_episode_job:
                                    alloc_bits_episode_job[jb.job_id] += serve
                                if jb.remaining_bits <= 0 and jb.done_time_s is None:
                                    jb.done_time_s = now_s + MINISLOT_S
                                    self.completed_jobs.append(jb)
                                    self.active_jobs.pop(jb.job_id, None)

                if ul_mode == "proactive":
                    qcrit_cells_used[(ci, "UL")] += q_crit_ul

# EPISODE KPIs
        # 1) Job-based violation: job is violated if allocated bits < its demand bits.
        job_total_counts = {(s, l): 0 for s in SLICES for l in LINKS}
        job_viol_counts = {(s, l): 0 for s in SLICES for l in LINKS}

        for m in episode_job_meta:
            key = (m["slice"], m["link"])
            job_total_counts[key] += 1
            alloc = float(alloc_bits_episode_job.get(m["job_id"], 0.0))
            demand = float(m["demand_bits"])
            if alloc + 1e-9 < demand:
                job_viol_counts[key] += 1

        job_viol_rate_pct = {(s, l): 0.0 for s in SLICES for l in LINKS}
        for s in SLICES:
            for l in LINKS:
                tot = job_total_counts[(s, l)]
                vio = job_viol_counts[(s, l)]
                job_viol_rate_pct[(s, l)] = (100.0 * vio / tot) if tot > 0 else 0.0

        job_total_slice = {s: job_total_counts[(s, "DL")] + job_total_counts[(s, "UL")] for s in SLICES}
        job_viol_slice = {s: job_viol_counts[(s, "DL")] + job_viol_counts[(s, "UL")] for s in SLICES}
        job_viol_slice_pct = {
            s: (100.0 * job_viol_slice[s] / job_total_slice[s]) if job_total_slice[s] > 0 else 0.0
            for s in SLICES
        }
        # 2) Violation gap rate for violated jobs only:
        #    gap = abs((assign - demand)/demand)
        job_gap_sum = {(s, l): 0.0 for s in SLICES for l in LINKS}

        for m in episode_job_meta:
            key = (m["slice"], m["link"])
            alloc = float(alloc_bits_episode_job.get(m["job_id"], 0.0))
            demand = float(m["demand_bits"])
            if demand > 0 and (alloc + 1e-9 < demand):
                job_gap_sum[key] += abs((alloc - demand) / demand)

        job_gap_rate = {(s, l): 0.0 for s in SLICES for l in LINKS}
        for s in SLICES:
            for l in LINKS:
                vio = job_viol_counts[(s, l)]
                job_gap_rate[(s, l)] = (job_gap_sum[(s, l)] / vio) if vio > 0 else 0.0

        job_gap_slice = {}
        for s in SLICES:
            vio_tot = job_viol_slice[s]
            gap_sum_tot = job_gap_sum[(s, "DL")] + job_gap_sum[(s, "UL")]
            job_gap_slice[s] = (gap_sum_tot / vio_tot) if vio_tot > 0 else 0.0

        # Throughput in kbps (split DL/UL and per-slice)
        thr_kbps = {}
        for s in SLICES:
            for l in LINKS:
                thr_kbps[(s,l)] = (served_bits[(s,l)] / EPISODE_S) / 1000.0  # bits/s -> kbps

        t_end = time.perf_counter()
        e_end_uj = _read_rapl_uj()
        runtime_ms = (t_end - t_start) * 1000.0
        energy_j = (e_end_uj - e_start_uj) / 1e6 if (not np.isnan(e_end_uj) and not np.isnan(e_start_uj)) else 0.0
        ram_mb = _read_rss_mb()

        row = {
            "step": step,
            "episode_start_s": episode_start_s,
            "scheduler": (
                "Reactive" if mode == "naive" else
                ("Proactive" if mode == "proactive" else
                 ("ProReactive_heuristic" if mode == "proreactive" else "OurPRISR_ProReactive_DRL"))
            ),
            "punctured_prbs_DL": punctured_prbs["DL"],
            "punctured_prbs_UL": punctured_prbs["UL"],
            "ces_count_episode": ces_count_episode,
            "emergency_mode": emergency_mode,
            "dropped_critical_jobs": dropped_critical_jobs,
            "drl_share_dl": drl_share_dl if mode == "proreactive_drl" else np.nan,
            "drl_share_ul": drl_share_ul if mode == "proreactive_drl" else np.nan,
            "drl_gate_dl": drl_gate_dl if mode == "proreactive_drl" else np.nan,
            "drl_gate_ul": drl_gate_ul if mode == "proreactive_drl" else np.nan,

            "runtime_ms": runtime_ms,
            "ram_mb": ram_mb,
            "energy_j": energy_j,
            "energy_available": 1 if (not np.isnan(e_end_uj) and not np.isnan(e_start_uj)) else 0,

            # demand/served/violation per slice+link
            "demand_bits_critical_DL": demand_bits_total_start[("critical","DL")],
            "served_bits_critical_DL": served_bits[("critical","DL")],
            "violation_rate_pct_critical_DL": job_viol_rate_pct[("critical","DL")],
            "violated_jobs_critical_DL": job_viol_counts[("critical","DL")],
            "violation_gap_rate_critical_DL": job_gap_rate[("critical","DL")],
            "total_jobs_critical_DL": job_total_counts[("critical","DL")],
            "thr_kbps_critical_DL": thr_kbps[("critical","DL")],

            "demand_bits_critical_UL": demand_bits_total_start[("critical","UL")],
            "served_bits_critical_UL": served_bits[("critical","UL")],
            "violation_rate_pct_critical_UL": job_viol_rate_pct[("critical","UL")],
            "violated_jobs_critical_UL": job_viol_counts[("critical","UL")],
            "violation_gap_rate_critical_UL": job_gap_rate[("critical","UL")],
            "total_jobs_critical_UL": job_total_counts[("critical","UL")],
            "thr_kbps_critical_UL": thr_kbps[("critical","UL")],

            "demand_bits_performance_DL": demand_bits_total_start[("performance","DL")],
            "served_bits_performance_DL": served_bits[("performance","DL")],
            "violation_rate_pct_performance_DL": job_viol_rate_pct[("performance","DL")],
            "violated_jobs_performance_DL": job_viol_counts[("performance","DL")],
            "violation_gap_rate_performance_DL": job_gap_rate[("performance","DL")],
            "total_jobs_performance_DL": job_total_counts[("performance","DL")],
            "thr_kbps_performance_DL": thr_kbps[("performance","DL")],

            "demand_bits_performance_UL": demand_bits_total_start[("performance","UL")],
            "served_bits_performance_UL": served_bits[("performance","UL")],
            "violation_rate_pct_performance_UL": job_viol_rate_pct[("performance","UL")],
            "violated_jobs_performance_UL": job_viol_counts[("performance","UL")],
            "violation_gap_rate_performance_UL": job_gap_rate[("performance","UL")],
            "total_jobs_performance_UL": job_total_counts[("performance","UL")],
            "thr_kbps_performance_UL": thr_kbps[("performance","UL")],

            # totals per slice
            "violation_rate_pct_critical_total": job_viol_slice_pct["critical"],
            "violation_rate_pct_performance_total": job_viol_slice_pct["performance"],
            "violated_jobs_critical_total": job_viol_slice["critical"],
            "violation_gap_rate_critical_total": job_gap_slice["critical"],
            "violated_jobs_performance_total": job_viol_slice["performance"],
            "violation_gap_rate_performance_total": job_gap_slice["performance"],
            "total_jobs_critical_total": job_total_slice["critical"],
            "total_jobs_performance_total": job_total_slice["performance"],
            "backlog_jobs_end_episode": len(self.active_jobs),
        }

# reward
        if mode == "proreactive_drl":
            demand_crit = demand_bits_total_start[("critical", "DL")] + demand_bits_total_start[("critical", "UL")]
            demand_perf = demand_bits_total_start[("performance", "DL")] + demand_bits_total_start[("performance", "UL")]
            served_crit = served_bits[("critical", "DL")] + served_bits[("critical", "UL")]
            served_perf = served_bits[("performance", "DL")] + served_bits[("performance", "UL")]

            thr_sat_crit = 1.0 if demand_crit <= 0 else min(served_crit / max(demand_crit, 1e-9), 1.0)
            thr_sat_perf = 1.0 if demand_perf <= 0 else min(served_perf / max(demand_perf, 1e-9), 1.0)

            done_jobs_ep = [
                jb for jb in self.completed_jobs
                if jb.done_time_s is not None and episode_start_s <= jb.done_time_s < (episode_start_s + EPISODE_S + 1e-12)
            ]
            crit_delays = [
                (jb.done_time_s - jb.arrival_time_s) * 1000.0
                for jb in done_jobs_ep if jb.slice_name == "critical"
            ]
            perf_delays = [
                (jb.done_time_s - jb.arrival_time_s) * 1000.0
                for jb in done_jobs_ep if jb.slice_name == "performance"
            ]

            if demand_crit <= 0:
                delay_sat_crit = 1.0
            elif len(crit_delays) == 0:
                delay_sat_crit = 0.0
            else:
                delay_sat_crit = min(DRL_DELAY_TARGET_CRIT_MS / max(float(np.mean(crit_delays)), 1e-9), 1.0)

            if demand_perf <= 0:
                delay_sat_perf = 1.0
            elif len(perf_delays) == 0:
                delay_sat_perf = 0.0
            else:
                delay_sat_perf = min(DRL_DELAY_TARGET_PERF_MS / max(float(np.mean(perf_delays)), 1e-9), 1.0)

            score_critical = 0.5 * (thr_sat_crit + delay_sat_crit)
            score_performance = 0.5 * (thr_sat_perf + delay_sat_perf)
            reward = 0.7 * score_critical + 0.3 * score_performance
        else:
            # legacy reward for non-DRL modes
            served_total = sum(served_bits.values())
            viol_total_pct = 0.5*(job_viol_slice_pct["critical"] + job_viol_slice_pct["performance"])
            backlog_bits = sum(j.remaining_bits for j in self.active_jobs.values())
            reward = (served_total / 1e6) - 0.05*(viol_total_pct) - 0.001*(backlog_bits/1e6)

        for k, v in list(row.items()):
            if isinstance(v, (float, np.floating)):
                row[k] = round(float(v), 7)
        self.episode_rows.append(row)
        self.ptr += 1
        self._prepared = None
        self.prev_pred_ci_link = {}
        self.obs_ema_ci_link = {}
        self.prev_crit_jobs_ci_link = {}
        return row, float(reward)

    def export_job_metrics(self) -> pd.DataFrame:
        rows = []
        for jb in self.completed_jobs:
            if jb.done_time_s is None:
                continue
            delay_ms = (jb.done_time_s - jb.arrival_time_s) * 1000.0
            rows.append({
                "train_id": jb.train_id,
                "app": jb.app_name,
                "link": jb.link,
                "slice": jb.slice_name,
                "seq": jb.job_id[3],
                "arrival_time_s": jb.arrival_time_s,
                "first_served_time_s": jb.first_served_time_s,
                "done_time_s": jb.done_time_s,
                "drop_time_s": np.nan,
                "dropped": 0,
                "delay_ms": delay_ms,
                "total_bits": jb.total_bits,
            })

        for jb in self.dropped_jobs:
            rows.append({
                "train_id": jb.train_id,
                "app": jb.app_name,
                "link": jb.link,
                "slice": jb.slice_name,
                "seq": jb.job_id[3],
                "arrival_time_s": jb.arrival_time_s,
                "first_served_time_s": jb.first_served_time_s,
                "done_time_s": np.nan,
                "drop_time_s": jb.dropped_time_s,
                "dropped": 1,
                "delay_ms": np.nan,
                "total_bits": jb.total_bits,
            })
        return pd.DataFrame(rows)

    def export_open_jobs_end(self) -> pd.DataFrame:
        if self.steps:
            sim_end_s = (self.steps[min(self.ptr, len(self.steps)-1)] + 1) * EPISODE_S
        else:
            sim_end_s = 0.0
        rows = []
        for jb in self.active_jobs.values():
            rows.append({
                "train_id": jb.train_id,
                "app": jb.app_name,
                "link": jb.link,
                "slice": jb.slice_name,
                "seq": jb.job_id[3],
                "arrival_time_s": jb.arrival_time_s,
                "first_served_time_s": jb.first_served_time_s,
                "age_s_at_end": sim_end_s - jb.arrival_time_s,
                "remaining_bits": jb.remaining_bits,
                "total_bits": jb.total_bits,
            })
        return pd.DataFrame(rows)

def _compute_start_ptr(st: SimulatorState, forecaster: Optional[STGMFiLMForecaster]) -> int:
    start_ptr = 0
    if forecaster is not None:
        while start_ptr < len(st.steps):
            ep_s = st.steps[start_ptr] * EPISODE_S
            t_idx = forecaster.time_to_idx.get(float(ep_s), None)
            if t_idx is not None and t_idx >= forecaster.history - 1:
                break
            start_ptr += 1
    return start_ptr


def _policy_display_name(policy: str) -> str:
    return {
        "naive": "Reactive",
        "proactive": "Proactive",
        "proreactive": "ProReactive_heuristic",
        "proreactive_drl": "OurPRISR_ProReactive_DRL",
    }.get(policy, policy)


def _create_progress_handle() -> Any:
    if display is None:
        return None
    try:
        return display("", display_id=True)
    except Exception:
        return None


def _update_progress(handle: Any, message: str) -> None:
    if handle is not None:
        try:
            handle.update(message)
            return
        except Exception:
            pass
    print(message, flush=True)


def run_policy(
    final_df: pd.DataFrame,
    train_store: TrainStore,
    forecaster: Optional[STGMFiLMForecaster],
    policy: str,
    drl_model: Optional[Any] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    st = SimulatorState(final_df, train_store, forecaster)
    st.reset()
    st.ptr = _compute_start_ptr(st, forecaster)

    policy_name = _policy_display_name(policy)
    total_episodes = max(0, len(st.steps) - st.ptr)
    progress_handle = _create_progress_handle()
    completed_episodes = 0
    _update_progress(progress_handle, f"[{policy_name}] progress: 0/{total_episodes} episodes completed")

    while not st.done():
        obs = st.prepare_episode().astype(np.float32)
        if policy == "proreactive_drl":
            if drl_model is None:
                raise ValueError("drl_model is required for proreactive_drl policy")
            action, _ = drl_model.predict(obs, deterministic=True)
            st.run_prepared_episode(mode="proreactive_drl", action=np.asarray(action, dtype=np.float32))
        else:
            st.run_prepared_episode(mode=policy)
        completed_episodes += 1
        _update_progress(progress_handle, f"[{policy_name}] progress: {completed_episodes}/{total_episodes} episodes completed")

    episode_df = pd.DataFrame(st.episode_rows)
    job_df = st.export_job_metrics()
    open_df = st.export_open_jobs_end()
    return episode_df, job_df, open_df


if PPO is not None:
    class EpisodeRewardCallback(BaseCallback):
        def __init__(self, total_episodes: int, print_every: int = 1, verbose: int = 0):
            super().__init__(verbose)
            self.total_episodes = int(max(1, total_episodes))
            self.print_every = int(max(1, print_every))
            self.episode_rewards: List[float] = []
            self.episode_idx = 0

        def _on_step(self) -> bool:
            infos = self.locals.get("infos", [])
            for info in infos:
                ep = info.get("episode") if isinstance(info, dict) else None
                if ep is None or "r" not in ep:
                    continue

                self.episode_idx += 1
                r = float(ep.get("r", 0.0))
                l = int(ep.get("l", 0))
                self.episode_rewards.append(r)

                if (self.episode_idx % self.print_every != 0) and (self.episode_idx != self.total_episodes):
                    continue

                logger = getattr(self.model, "logger", None)
                vals = getattr(logger, "name_to_value", {}) if logger is not None else {}
                loss = vals.get("train/loss", None) if isinstance(vals, dict) else None
                vloss = vals.get("train/value_loss", None) if isinstance(vals, dict) else None
                pgloss = vals.get("train/policy_gradient_loss", None) if isinstance(vals, dict) else None

                if loss is None:
                    print(f"[OurPRISR_ProReactive_DRL][Train] episode={self.episode_idx} reward={r:.4f} len={l}")
                else:
                    loss_f = float(loss)
                    vloss_f = float(vloss) if vloss is not None else float("nan")
                    pgloss_f = float(pgloss) if pgloss is not None else float("nan")
                    print(
                        f"[OurPRISR_ProReactive_DRL][Train] episode={self.episode_idx} "
                        f"reward={r:.4f} len={l} loss={loss_f:.5f} value_loss={vloss_f:.5f} pg_loss={pgloss_f:.5f}"
                    )

            return True
else:
    class EpisodeRewardCallback:
        def __init__(self, total_episodes: int, print_every: int = 1, verbose: int = 0):
            self.episode_rewards: List[float] = []
            self.episode_idx = 0


class ProReactiveDRLEnv(gym.Env if gym is not None else object):
    def __init__(
        self,
        final_df: pd.DataFrame,
        train_store: TrainStore,
        forecaster: Optional[STGMFiLMForecaster],
        max_steps_per_episode: Optional[int] = None,
    ):
        if gym is None or spaces is None:
            raise ImportError("gymnasium is required for OurPRISR_ProReactive_DRL")
        super().__init__()
        self.final_df = final_df
        self.train_store = train_store
        self.forecaster = forecaster
        self.max_steps_per_episode = int(max_steps_per_episode) if max_steps_per_episode is not None else None

        tmp_state = SimulatorState(final_df, train_store, forecaster)
        self.obs_dim = tmp_state.obs_dim

        self.action_space = spaces.Box(low=0.0, high=1.0, shape=(4,), dtype=np.float32)
        self.observation_space = spaces.Box(low=0.0, high=np.float32(20.0), shape=(self.obs_dim,), dtype=np.float32)

        self.st: Optional[SimulatorState] = None
        self.steps_in_episode = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.st = SimulatorState(self.final_df, self.train_store, self.forecaster)
        self.st.reset()
        self.st.ptr = _compute_start_ptr(self.st, self.forecaster)
        self.steps_in_episode = 0

        if self.st.done():
            obs = np.zeros(self.obs_dim, dtype=np.float32)
        else:
            obs = self.st.prepare_episode().astype(np.float32)
        return obs, {}

    def step(self, action):
        assert self.st is not None, "Call reset() before step()"

        if self.st.done():
            obs = np.zeros(self.obs_dim, dtype=np.float32)
            return obs, 0.0, True, False, {}

        _row, reward = self.st.run_prepared_episode(mode="proreactive_drl", action=np.asarray(action, dtype=np.float32))
        self.steps_in_episode += 1

        terminated = self.st.done()
        truncated = (
            (self.max_steps_per_episode is not None)
            and (self.steps_in_episode >= self.max_steps_per_episode)
            and (not terminated)
        )

        if terminated or truncated:
            obs = np.zeros(self.obs_dim, dtype=np.float32)
        else:
            obs = self.st.prepare_episode().astype(np.float32)

        return obs, float(reward), bool(terminated), bool(truncated), {}


def train_proreactive_drl(
    final_df: pd.DataFrame,
    train_store: TrainStore,
    forecaster: Optional[STGMFiLMForecaster],
    out_dir: Path,
    train_episodes: int = DRL_TRAIN_EPISODES,
) -> Path:
    if PPO is None or gym is None:
        raise ImportError("stable-baselines3 and gymnasium are required for OurPRISR_ProReactive_DRL")

    tmp_state = SimulatorState(final_df, train_store, forecaster)
    start_ptr = _compute_start_ptr(tmp_state, forecaster)
    steps_per_sim = max(1, len(tmp_state.steps) - start_ptr)
    steps_per_episode = min(steps_per_sim, DRL_MAX_TRAIN_STEPS_PER_EPISODE)
    total_timesteps = int(max(1, train_episodes) * steps_per_episode)

    env = ProReactiveDRLEnv(final_df, train_store, forecaster, max_steps_per_episode=DRL_MAX_TRAIN_STEPS_PER_EPISODE)
    env_train = Monitor(env) if Monitor is not None else env

    n_steps = int(max(2, min(64, steps_per_episode)))
    batch_size = int(min(64, n_steps))
    model = PPO(
        "MlpPolicy",
        env_train,
        verbose=1,
        n_steps=n_steps,
        batch_size=batch_size,
        gamma=0.99,
        learning_rate=3e-4,
    )

    print(f"[OurPRISR_ProReactive_DRL][Train] target_episodes={train_episodes} steps_per_episode={steps_per_episode} (full_sim_steps={steps_per_sim}) total_timesteps={total_timesteps}")
    cb = EpisodeRewardCallback(total_episodes=train_episodes, print_every=1)
    model.learn(total_timesteps=total_timesteps, callback=cb, log_interval=1)

    DRL_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    model.save(DRL_MODEL_PATH)

    if len(cb.episode_rewards) > 0:
        plt.figure()
        plt.plot(np.arange(1, len(cb.episode_rewards) + 1), cb.episode_rewards)
        plt.xlabel("Training Episode")
        plt.ylabel("Episode Reward")
        plt.title("OurPRISR_ProReactive_DRL Training Rewards")
        plt.tight_layout()
        plt.savefig(out_dir / DRL_TRAINING_REWARD_PLOT, dpi=160)

    return DRL_MODEL_PATH


# ========================= MAIN =========================
def _write_empty_outputs(out_dir: Path, suffix: str) -> None:
    pd.DataFrame().to_csv(out_dir / f"episode_metrics_{suffix}.csv", index=False)
    pd.DataFrame().to_csv(out_dir / f"job_metrics_{suffix}.csv", index=False)
    pd.DataFrame().to_csv(out_dir / f"open_jobs_end_{suffix}.csv", index=False)

def main():
    if not DATASET_BUNDLE_ROOT.exists():
        raise FileNotFoundError(f"Missing dataset bundle root: {DATASET_BUNDLE_ROOT}")
    if not SCENARIO_ROOT.exists():
        raise FileNotFoundError(f"Missing selected scenario folder: {SCENARIO_ROOT}")
    if not FINAL_DATASET_CSV.exists():
        raise FileNotFoundError(f"Missing scenario graph CSV: {FINAL_DATASET_CSV}")
    if not TRAINS_FOLDER.exists():
        raise FileNotFoundError(f"Missing scenario input_trains folder: {TRAINS_FOLDER}")
    if not CELL_GNB_PATH.exists():
        raise FileNotFoundError(f"Missing cell-gNB mapping CSV: {CELL_GNB_PATH}")

    out_dir = Path("sim_outputs_v5_compare") / SCENARIO_NAME
    out_dir.mkdir(parents=True, exist_ok=True)

    final_df = load_presence_from_final_dataset(FINAL_DATASET_CSV)
    train_store = TrainStore(TRAINS_FOLDER)

    forecaster = None
    if USE_FORECAST and FORECAST_CKPT.exists():
        forecaster = STGMFiLMForecaster(FORECAST_CKPT, FINAL_DATASET_CSV, CELL_GNB_PATH, device=DEVICE)
        print(f"Loaded forecaster from: {FORECAST_CKPT}")
    else:
        print("Forecaster not used (missing checkpoint or disabled). Pro-Reactive will use pred=0s.")

    # Energy counter check (RAPL).
    if np.isnan(_read_rapl_uj()):
        print("Warning: RAPL energy counter is not readable; energy_j will be 0.0. Run contrib3/setup_rapl_perms.sh, then restart kernel/session.")

    approaches_run = []

    if RUN_APPROACHES.get("Reactive", False):
        ep_reac, job_reac, open_reac = run_policy(final_df, train_store, forecaster, policy="naive")
        ep_reac.to_csv(out_dir / "episode_metrics_reactive.csv", index=False)
        job_reac.to_csv(out_dir / "job_metrics_reactive.csv", index=False)
        open_reac.to_csv(out_dir / "open_jobs_end_reactive.csv", index=False)
        approaches_run.append("Reactive")
        print(f"[Reactive] episodes={len(ep_reac)} jobs_done={len(job_reac)} open_jobs={len(open_reac)}")
    else:
        _write_empty_outputs(out_dir, "reactive")
        print("[Reactive] skipped")

    if RUN_APPROACHES.get("Proactive", False):
        ep_heur, job_heur, open_heur = run_policy(final_df, train_store, forecaster, policy="proactive")
        ep_heur.to_csv(out_dir / "episode_metrics_proactive_heur.csv", index=False)
        job_heur.to_csv(out_dir / "job_metrics_proactive_heur.csv", index=False)
        open_heur.to_csv(out_dir / "open_jobs_end_proactive_heur.csv", index=False)
        approaches_run.append("Proactive")
        print(f"[Proactive] episodes={len(ep_heur)} jobs_done={len(job_heur)} open_jobs={len(open_heur)}")
    else:
        _write_empty_outputs(out_dir, "proactive_heur")
        print("[Proactive] skipped")

    if RUN_APPROACHES.get("ProReactive_heuristic", False):
        ep_pro, job_pro, open_pro = run_policy(final_df, train_store, forecaster, policy="proreactive")
        ep_pro.to_csv(out_dir / "episode_metrics_proreactive.csv", index=False)
        job_pro.to_csv(out_dir / "job_metrics_proreactive.csv", index=False)
        open_pro.to_csv(out_dir / "open_jobs_end_proreactive.csv", index=False)
        approaches_run.append("ProReactive_heuristic")
        print(f"[ProReactive_heuristic] episodes={len(ep_pro)} jobs_done={len(job_pro)} open_jobs={len(open_pro)}")
    else:
        _write_empty_outputs(out_dir, "proreactive")
        print("[ProReactive_heuristic] skipped")

    if RUN_APPROACHES.get("OurPRISR_ProReactive_DRL", False):
        if PPO is None:
            raise ImportError("Install gymnasium + stable-baselines3 to use OurPRISR_ProReactive_DRL")

        if IF_TRAINING:
            print(f"[OurPRISR_ProReactive_DRL] Training for {DRL_TRAIN_EPISODES} episodes...")
            model_path = train_proreactive_drl(final_df, train_store, forecaster, out_dir, train_episodes=DRL_TRAIN_EPISODES)
            print(f"[OurPRISR_ProReactive_DRL] Saved model: {model_path}")
        else:
            model_path = DRL_MODEL_PATH
            if not model_path.exists():
                raise FileNotFoundError(f"IF_TRAINING=False but model not found: {model_path}")
            print(f"[OurPRISR_ProReactive_DRL] IF_TRAINING=False -> loading model: {model_path}")

        drl_model = PPO.load(model_path)
        ep_drl, job_drl, open_drl = run_policy(final_df, train_store, forecaster, policy="proreactive_drl", drl_model=drl_model)
        ep_drl.to_csv(out_dir / "episode_metrics_proreactive_drl.csv", index=False)
        job_drl.to_csv(out_dir / "job_metrics_proreactive_drl.csv", index=False)
        open_drl.to_csv(out_dir / "open_jobs_end_proreactive_drl.csv", index=False)
        approaches_run.append("OurPRISR_ProReactive_DRL")
        print(f"[OurPRISR_ProReactive_DRL] episodes={len(ep_drl)} jobs_done={len(job_drl)} open_jobs={len(open_drl)}")
    else:
        _write_empty_outputs(out_dir, "proreactive_drl")
        print("[OurPRISR_ProReactive_DRL] skipped")

    print("Approaches run:", ", ".join(approaches_run) if approaches_run else "None")

    print("\nSaved outputs to:", out_dir.resolve())
    print("Saved training reward plot:", out_dir / DRL_TRAINING_REWARD_PLOT)
    print("Simulation done. Run the separate plotting cell to generate comparison figures.")


if __name__ == "__main__":
    main()


/home/me/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
2026-06-12 00:57:29.938852: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-12 00:57:29.938890: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-12 00:57:29.939878: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been regis

Loaded forecaster from: forecasting_results_missingData/checkpoints/STGdata_C_STG_MFilmFormer_best.pt


'[Reactive] progress: 1586/1586 episodes completed'

[Reactive] episodes=1586 jobs_done=810344 open_jobs=90
[Proactive] skipped
[ProReactive_heuristic] skipped
[OurPRISR_ProReactive_DRL] IF_TRAINING=False -> loading model: saved_models/proreactive_drl_ppo.zip


'[OurPRISR_ProReactive_DRL] progress: 849/1586 episodes completed'

KeyboardInterrupt: 

## Plot defintion and run

In [2]:
# Plot-only cell (re-runnable without re-running simulation)
from pathlib import Path
import importlib.util

scenario_name = globals().get("SCENARIO_NAME", "STGdata_A")

# Use the local PRIS-R output folder and plotting helper.
out_dir = Path("sim_outputs_v5_compare") / scenario_name
plot_mod_path = Path("plot_from_csv.py")
if not plot_mod_path.exists():
    raise FileNotFoundError(f"Missing plotting helper: {plot_mod_path}")

spec = importlib.util.spec_from_file_location("plot_from_csv", plot_mod_path)
plot_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_mod)

# Robust CSV loading (handles empty files and missing columns).
reac_ep = plot_mod.read_csv_safe(out_dir / "episode_metrics_reactive.csv", plot_mod.EPISODE_COLUMNS)
reac_jobs = plot_mod.read_csv_safe(out_dir / "job_metrics_reactive.csv", plot_mod.JOB_COLUMNS)
reac_open = plot_mod.read_csv_safe(out_dir / "open_jobs_end_reactive.csv", plot_mod.OPEN_COLUMNS)

heur_ep = plot_mod.read_csv_safe(out_dir / "episode_metrics_proactive_heur.csv", plot_mod.EPISODE_COLUMNS)
heur_jobs = plot_mod.read_csv_safe(out_dir / "job_metrics_proactive_heur.csv", plot_mod.JOB_COLUMNS)
heur_open = plot_mod.read_csv_safe(out_dir / "open_jobs_end_proactive_heur.csv", plot_mod.OPEN_COLUMNS)

pro_ep = plot_mod.read_csv_safe(out_dir / "episode_metrics_proreactive.csv", plot_mod.EPISODE_COLUMNS)
pro_jobs = plot_mod.read_csv_safe(out_dir / "job_metrics_proreactive.csv", plot_mod.JOB_COLUMNS)
pro_open = plot_mod.read_csv_safe(out_dir / "open_jobs_end_proreactive.csv", plot_mod.OPEN_COLUMNS)

drl_ep = plot_mod.read_csv_safe(out_dir / "episode_metrics_proreactive_drl.csv", plot_mod.EPISODE_COLUMNS)
drl_jobs = plot_mod.read_csv_safe(out_dir / "job_metrics_proreactive_drl.csv", plot_mod.JOB_COLUMNS)
drl_open = plot_mod.read_csv_safe(out_dir / "open_jobs_end_proreactive_drl.csv", plot_mod.OPEN_COLUMNS)

# Generate all comparison plots.
plot_mod.plot_compare(
    reac_ep, reac_jobs, reac_open,
    heur_ep, heur_jobs, heur_open,
    pro_ep, pro_jobs, pro_open,
    drl_ep, drl_jobs, drl_open,
    out_dir,
)

print("Loaded simulation CSVs from:", out_dir.resolve())
print("Saved comparison plots to:", out_dir.resolve())


Loaded simulation CSVs from: /home/me/contrib3/PRIS-R/sim_outputs_v5_compare/STGdata_C
Saved comparison plots to: /home/me/contrib3/PRIS-R/sim_outputs_v5_compare/STGdata_C
